<a href="https://colab.research.google.com/github/peter-adepoju/math-neuro-ml/blob/main/week4/lesson15/Lesson15_Exercises_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lesson 15 — Exercises**
## Multidimensional Optimisation: Valleys, Scaling and Momentum

**Course:** MAMBA: Mathematical Sciences for Brain Data\
**Week:** Week 4 — Calculus II\
**Lesson:** Lesson 15

---
This notebook has **7 required exercises worth 100 points total**, plus one **bonus
challenge** (extra credit, not counted in the 100). Work through them in order — later
exercises reuse functions and data from earlier ones.

**How to use this notebook:**
- Cells marked `# YOUR CODE HERE` are where you write code. Remove the `raise
  NotImplementedError` line when you add your answer.
- Cells marked **Written response** are where you type a short answer in the Markdown cell
  provided (double-click the cell to edit it).
- Cells marked **Self-check** run lightweight checks on your work. Passing a self-check is a
  good sign, but it does not guarantee full credit — it cannot check written reasoning.
- Do not change the random seed or provided data unless an exercise explicitly asks you to.

**Point allocation**

| Exercise | Topic | Points |
|---|---|---|
| 1 | A trustworthy vector optimiser | 15 |
| 2 | Eigenmodes, learning rate and stability | 15 |
| 3 | Conditioning, scaling and preconditioning | 15 |
| 4 | Momentum | 15 |
| 5 | Optimiser diagnosis clinic | 15 |
| 6 | Regularisation and validation monitoring | 10 |
| 7 | Real data: fitting a speed-tuning curve by gradient descent | 15 |
| **Total** | | **100** |
| Bonus | A backtracking line search | extra credit |

In [ ]:
# SETUP CELL -- run this first.
import numpy as np
import matplotlib.pyplot as plt

PALETTE = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']

def num_grad(f, p, h=1e-6):
    # Central-difference gradient with a scale-aware step, shape (p,).
    p = np.asarray(p, float); g = np.zeros_like(p)
    for i in range(p.size):
        e = np.zeros_like(p); e[i] = h * (1 + abs(p[i]))
        g[i] = (f(p + e) - f(p - e)) / (2 * e[i])
    return g

print("Setup complete.")

---
## Exercise 1 — A Trustworthy Vector Optimiser (15 points)

Throughout this exercise, $L(\boldsymbol\theta) = \tfrac12\boldsymbol\theta^{\mathsf T}\mathbf{H}\boldsymbol\theta - \mathbf{b}^{\mathsf T}\boldsymbol\theta$ with $\mathbf{H} = \begin{pmatrix}3 & 1\\ 1 & 2\end{pmatrix}$ and $\mathbf{b} = (1, 1)^{\mathsf T}$.

**1.1 (4 points, written response, no code).** (a) Give $\nabla L$ and the minimiser $\boldsymbol\theta^\star$. (b) From $\boldsymbol\theta_0 = \mathbf{0}$ with $\eta = 0.2$, compute $\boldsymbol\theta_1$ and $\boldsymbol\theta_2$ by hand. (c) Why must both components of $\boldsymbol\theta_1$ be computed from $\nabla L(\boldsymbol\theta_0)$, and what would change here if $\theta_2$ were updated using the already-updated $\theta_1$?

**Your answer:**

*(replace this line with your answer)*

In [ ]:
# 1.2 (7 points): Write gd_run(objective, gradient, theta0, eta, max_steps=1000, grad_tol=1e-8, diverge_at=1e6) that
#  - copies theta0 to a float array (shape (p,)) and checks eta > 0;
#  - stores aligned histories INCLUDING iteration 0: 'theta' (k + 1, p), 'obj' (k + 1,), 'grad_norm' (k + 1,);
#  - stops with reason 'gradient tolerance' when ||grad|| <= grad_tol (checked before each step), 'diverged' when theta is not
#    finite or ||theta|| > diverge_at, otherwise 'max_steps' (never call that converged);
#  - updates all components simultaneously and stores copies.
# Return a dict with keys 'theta', 'obj', 'grad_norm', 'stop_reason', 'n_iter'. Run it on L above from 0 with eta = 0.2 and store
# the result in res1.

H1 = np.array([[3.0, 1.0], [1.0, 2.0]]); b1 = np.array([1.0, 1.0])
L1 = lambda th: 0.5 * th @ H1 @ th - b1 @ th
g1 = lambda th: H1 @ th - b1

# YOUR CODE HERE
res1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.2
assert res1 is not None, "Set res1."
k_ = res1['n_iter']
assert res1['theta'].shape == (k_ + 1, 2) and res1['obj'].shape == (k_ + 1,) and res1['grad_norm'].shape == (k_ + 1,), "Histories must include iteration 0 and be aligned."
assert np.allclose(res1['theta'][0], 0) and np.allclose(res1['theta'][1], [0.2, 0.2]), "The first step should match the hand calculation."
assert res1['stop_reason'] == 'gradient tolerance' and np.allclose(res1['theta'][-1], [0.2, 0.4], atol=1e-7), "Should converge to (0.2, 0.4)."
assert len(np.unique(res1['theta'].round(12), axis=0)) > 5, "Stored iterates must be copies, not references to one array."
r_short = gd_run(L1, g1, np.zeros(2), 0.2, max_steps=3)
assert r_short['stop_reason'] == 'max_steps' and r_short['n_iter'] == 3, "Running out of steps is 'max_steps', not convergence."
assert gd_run(L1, g1, np.zeros(2), 0.7, max_steps=500)['stop_reason'] == 'diverged', "eta = 0.7 > 2/lambda_max should diverge."
print("Self-check passed.")

In [ ]:
# 1.3 (4 points): Two candidate gradient functions are given for the objective L1. Use a central-difference check (num_grad) at
# three random points (rng = np.random.default_rng(13), points ~ normal) to find which one is wrong. Store the largest relative
# error of each in gerr1 = {'A': ..., 'B': ...} and the name of the wrong one in bad1.

gradA = lambda th: H1 @ th - b1
gradB = lambda th: np.array([3 * th[0] + 2 * th[1] - 1, th[0] + 2 * th[1] - 1])

# YOUR CODE HERE
gerr1 = bad1 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 1.3
assert gerr1 is not None and bad1 is not None, "Set gerr1 and bad1."
assert set(gerr1) == {'A', 'B'} and min(gerr1.values()) < 1e-6 and max(gerr1.values()) > 1e-3, "One gradient should pass and one fail."
print("Self-check passed.")

---
## Exercise 2 — Eigenmodes, Learning Rate and Stability (15 points)

Use the quadratic of Exercise 1 ($\mathbf{H}$, $\mathbf{b}$, $\boldsymbol\theta^\star = (0.2, 0.4)$).

**2.1 (5 points, written response).** (a) Derive the error recursion $\mathbf{e}_{k+1} = (\mathbf{I} - \eta\mathbf{H})\mathbf{e}_k$ and its eigen-coordinate form. (b) Find the eigenvalues of $\mathbf{H}$ exactly, and hence the interval of learning rates for which gradient descent converges from every start. (c) For $\eta = 0.5$, predict whether the path will oscillate, and which mode decides how fast the error shrinks at late iterations.

**Your answer:**

*(replace this line with your answer)*

In [ ]:
# 2.2 (6 points): For each eta in etas2 = [0.1, 0.3, 0.5, 0.5528, 0.6], compute the per-mode multipliers 1 - eta lambda_i, the
# spectral radius rho, and the predicted status ('converges' if rho < 1, 'marginal' if |rho - 1| < 1e-3, 'diverges' otherwise).
# Run gd_run for 300 steps from theta0 = (2, -1) and record the stop reason and final distance to theta*. Store a list of dicts
# (keys: eta, mult, rho, predicted, stop_reason, dist) in table2 and print it.

etas2 = [0.1, 0.3, 0.5, 0.5528, 0.6]
theta_star1 = np.linalg.solve(H1, b1)

# YOUR CODE HERE
table2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.2
assert table2 is not None, "Set table2."
assert len(table2) == 5 and all({'eta', 'mult', 'rho', 'predicted', 'stop_reason', 'dist'} <= set(r) for r in table2), "Five rows with the required keys."
preds = [r['predicted'] for r in table2]
assert preds == ['converges', 'converges', 'converges', 'marginal', 'diverges'], "Check the spectral radii against 2 / lambda_max = 0.5528."
assert table2[-1]['stop_reason'] == 'diverged' and table2[0]['dist'] < 1e-8, "Observed outcomes should match the theory."
print("Self-check passed.")

In [ ]:
# 2.3 (4 points): (i) With eta = 0.4, verify numerically that the eigen-coordinates of the error shrink by exactly 1 - eta lambda_i
# per step (store the largest deviation over 20 steps in dev2, excluding |z| < 1e-12). (ii) Scan eta on np.linspace(0.01, 0.55, 541)
# and store the eta minimising rho(I - eta H) in eta_best2, the minimum rho in rho_best2, and compare with 2 / (lmax + lmin) and
# (kappa - 1) / (kappa + 1).

# YOUR CODE HERE
dev2 = eta_best2 = rho_best2 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 2.3
assert dev2 is not None and eta_best2 is not None and rho_best2 is not None, "Set dev2, eta_best2 and rho_best2."
assert dev2 < 1e-8, "The eigen-coordinate ratios should equal 1 - eta lambda_i."
assert abs(eta_best2 - 0.4) < 2e-3 and abs(rho_best2 - (np.sqrt(5) / 5)) < 2e-3, "Compare with 2/(lmax + lmin) and (kappa - 1)/(kappa + 1)."
print("Self-check passed.")

---
## Exercise 3 — Conditioning, Scaling and Preconditioning (15 points)

In [ ]:
# 3.1 (5 points): Write cond_pd(H) returning (kappa, message): kappa = lambda_max / lambda_min (from np.linalg.eigvalsh) only if
# H is symmetric and all eigenvalues are > 1e-12 * |lambda_max|; otherwise kappa = None with message 'singular (flat direction)'
# or 'indefinite (no minimum)'. Apply it to the five Hessians in Hs3 and store the results in cond3 (dict name -> (kappa, message)).

R = np.array([[np.cos(0.6), -np.sin(0.6)], [np.sin(0.6), np.cos(0.6)]])
Hs3 = {'circular': np.diag([2.0, 2.0]), 'elongated': np.diag([1.0, 50.0]), 'rotated': R @ np.diag([1.0, 50.0]) @ R.T,
       'singular': np.array([[1.0, 1.0], [1.0, 1.0]]), 'indefinite': np.diag([1.0, -1.0])}

# YOUR CODE HERE
cond3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.1
assert cond3 is not None, "Set cond3."
assert abs(cond3['circular'][0] - 1) < 1e-9 and abs(cond3['elongated'][0] - 50) < 1e-9 and abs(cond3['rotated'][0] - 50) < 1e-8, "Rotation does not change kappa."
assert cond3['singular'][0] is None and cond3['indefinite'][0] is None, "kappa is undefined for singular or indefinite Hessians."
print("Self-check passed.")

In [ ]:
# 3.2 (5 points): For L(theta) = 0.5 theta^T H theta with H = diag(1, 50), choose a diagonal D so that D^T H D = I. Run plain gradient
# descent in theta (eta = 1/50) and gradient descent in z = D^{-1} theta (eta = 1), both from theta0 = (3, 1), with gd_run and
# grad_tol 1e-8. Show that the z-run mapped back (theta = D z) equals PRECONDITIONED gradient descent theta <- theta - P grad L with
# P = D D^T (store the max abs difference of the two paths in diff3), and store the iteration counts in steps3 = (plain, scaled).

H3 = np.diag([1.0, 50.0])
L3 = lambda th: 0.5 * th @ H3 @ th
g3 = lambda th: H3 @ th

# YOUR CODE HERE
D3 = diff3 = steps3 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 3.2
assert D3 is not None and diff3 is not None and steps3 is not None, "Set D3, diff3 and steps3."
assert np.allclose(D3.T @ H3 @ D3, np.eye(2)), "D should make the transformed Hessian the identity."
assert diff3 < 1e-12 and steps3[1] < 5 < steps3[0], "Scaled GD = preconditioned GD, and it should be far faster here."
print("Self-check passed.")

**3.3 (5 points, written response).** (a) In 3.2 the scaling made the problem perfect because $\mathbf{H}$ was diagonal. What would diagonal scaling achieve for the **rotated** Hessian of 3.1, and why? (b) List what feature standardisation fixes and what it cannot fix. (c) Why must the means and SDs used for standardisation come from the training rows only, and what should be done with a feature whose training SD is zero?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 4 — Momentum (15 points)

Use the heavy-ball convention of the teaching notebook: $\mathbf{v}_{k+1} = \beta\mathbf{v}_k + \nabla L(\boldsymbol\theta_k)$, $\boldsymbol\theta_{k+1} = \boldsymbol\theta_k - \eta\mathbf{v}_{k+1}$, $\mathbf{v}_0 = \mathbf{0}$, on $L = \frac12\boldsymbol\theta^{\mathsf T}\operatorname{diag}(1, 100)\,\boldsymbol\theta$ from $\boldsymbol\theta_0 = (2, 1)$.

In [ ]:
# 4.1 (5 points): Write momentum_run(gradient, theta0, eta, beta, max_steps=5000, grad_tol=1e-8, diverge_at=1e6) returning
# (theta_history (k + 1, p), stop_reason, n_iter) with the same stop reasons as gd_run. Check that beta = 0 reproduces gd_run exactly
# (store the max abs path difference in eq4) and store the iteration counts for (eta = 0.01, beta = 0) and (eta = 0.01, beta = 0.9)
# in steps4.

H4 = np.diag([1.0, 100.0])
L4 = lambda th: 0.5 * th @ H4 @ th
g4 = lambda th: H4 @ th
th0_4 = np.array([2.0, 1.0])

# YOUR CODE HERE
eq4 = steps4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.1
assert eq4 is not None and steps4 is not None, "Set eq4 and steps4."
assert eq4 == 0.0, "beta = 0 must reproduce vanilla gradient descent exactly (same path, same length)."
assert steps4[1] < steps4[0] / 3, "On this ill-conditioned valley momentum should be several times faster at the same eta."
print("Self-check passed.")

In [ ]:
# 4.2 (6 points): Scan etas4 x betas4 below: for each pair store the iteration count (or np.inf if the run diverged or hit max_steps)
# in steps_grid4 (shape (len(etas4), len(betas4))). Store the best pair in best4 = (eta, beta, steps), and the best vanilla
# (beta = 0) count in best_vanilla4. Check every diverged pair against the theory: diverges iff eta * 100 > 2 (1 + beta); store the
# fraction of pairs where theory and outcome agree in agree4. Show steps_grid4 as an image (log colour scale).

etas4 = np.array([0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.035])
betas4 = np.array([0.0, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95])

# YOUR CODE HERE
steps_grid4 = best4 = best_vanilla4 = agree4 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 4.2
assert steps_grid4 is not None and best4 is not None and best_vanilla4 is not None and agree4 is not None, "Set steps_grid4, best4, best_vanilla4 and agree4."
assert np.shape(steps_grid4) == (7, 8), "One entry per (eta, beta) pair."
assert best4[2] < best_vanilla4 / 4 and best4[1] > 0, "Tuned momentum should beat the best vanilla step clearly."
assert agree4 == 1.0, "Divergence should match eta lambda_max > 2(1 + beta) exactly for this quadratic."
print("Self-check passed.")

**4.3 (4 points, written response).** (a) Explain, in terms of the sharp and flat directions, why momentum speeds up progress on this valley. (b) Why is "momentum with the same $\eta$ is faster" not a fair comparison, and what is? (c) Give two situations where momentum does not help or harms.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 5 — Optimiser Diagnosis Clinic (15 points)

A colleague ran four optimisers on the objective $L_5(\boldsymbol\theta) = \frac{1}{2n}\lVert\mathbf{X}\boldsymbol\theta - \mathbf{y}\rVert^2$ (data below, $n = 50$), each with one bug. You only receive the functions as black boxes: **do not read their code to diagnose; diagnose from evidence**, then check your diagnosis against the code. The correct gradient is $\frac1n\mathbf{X}^{\mathsf T}(\mathbf{X}\boldsymbol\theta - \mathbf{y})$.

In [ ]:
# PROVIDED -- do not modify. Four faulty optimisers; each returns (theta_history, reported_status).
rng5 = np.random.default_rng(5)
X5 = np.column_stack([np.ones(50), rng5.normal(size=50)])
y5 = X5 @ np.array([1.0, 2.0]) + rng5.normal(0, 0.3, 50)
L5 = lambda th: 0.5 * np.mean((X5 @ th - y5)**2)
g5 = lambda th: X5.T @ (X5 @ th - y5) / 50

def opt_A(theta0, eta=0.3, steps=40):
    th = np.array(theta0, float); hist = [th.copy()]
    for _ in range(steps):
        th = th + eta * g5(th); hist.append(th.copy())
    return np.array(hist), 'finished'
def opt_B(theta0, eta=0.3, steps=40):
    th = np.array(theta0, float); hist = [th.copy()]
    for _ in range(steps):
        th = th - eta * (X5.T @ (X5 @ th - y5)); hist.append(th.copy())
        if np.linalg.norm(th) > 1e8: break
    return np.array(hist), 'finished'
def opt_C(theta0, eta=0.3, steps=40):
    th = np.array(theta0, float); hist = [th]
    for _ in range(steps):
        th -= eta * g5(th); hist.append(th)
    return np.array(hist), 'converged'
def opt_D(theta0, eta=0.001, steps=40):
    th = np.array(theta0, float); hist = [th.copy()]
    for _ in range(steps):
        th = th - eta * g5(th); hist.append(th.copy())
    return np.array(hist), 'converged'
print("Four optimisers ready. Correct reference:", np.round(np.linalg.lstsq(X5, y5, rcond=None)[0], 4))

In [ ]:
# 5.1 (6 points): Run each optimiser from theta0 = (0, 0). For each, compute the evidence: objective at the start and the end, whether
# the objective increased on the first step, the number of DISTINCT stored iterates, the final gradient norm (with the correct g5),
# and the ratio between its first step and the correct first step -eta * g5(theta0) (norm of its step / norm of the correct step,
# with the eta each optimiser uses: 0.3, 0.3, 0.3, 0.001). Store a dict name -> dict of evidence in evidence5 and print a table.

opts5 = {'A': (opt_A, 0.3), 'B': (opt_B, 0.3), 'C': (opt_C, 0.3), 'D': (opt_D, 0.001)}

# YOUR CODE HERE
evidence5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.1
assert evidence5 is not None, "Set evidence5."
assert set(evidence5) == {'A', 'B', 'C', 'D'}, "Evidence for all four optimisers."
assert all({'L_start', 'L_end', 'up_first', 'distinct', 'grad_end'} <= set(e) for e in evidence5.values()), "Missing evidence fields."
print("Self-check passed.")

In [ ]:
# 5.2 (5 points): Diagnose each optimiser with one label from: 'sign error', 'sum/mean normalisation mismatch', 'history stores
# references (missing copy)', 'learning rate too small, false convergence claim'. Store diag5 = {'A': ..., 'B': ..., 'C': ..., 'D': ...}.
# Then write fixed_run(theta0) (correct gradient, eta = 0.3, simultaneous update, copies, honest status) and store its final theta
# in fixed5.

labels5 = ['sign error', 'sum/mean normalisation mismatch', 'history stores references (missing copy)', 'learning rate too small, false convergence claim']

# YOUR CODE HERE
diag5 = fixed5 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 5.2
assert diag5 is not None and fixed5 is not None, "Set diag5 and fixed5."
assert set(diag5) == {'A', 'B', 'C', 'D'} and sorted(diag5.values()) == sorted(labels5), "Use each label exactly once."
assert np.allclose(fixed5, np.linalg.lstsq(X5, y5, rcond=None)[0], atol=1e-8), "The repaired optimiser should reach the least-squares solution."
print("Self-check passed.")

**5.3 (4 points, written response).** For each optimiser, name the piece of evidence that identified its bug. Then explain why "just lower the learning rate" would have hidden one of these bugs without fixing it, and why "run more iterations" would have fixed none of them.

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 6 — Regularisation and Validation Monitoring (10 points)

A neuron's response is predicted from 40 stimulus features with only 60 training trials. Use the ridge objective

$$L_{\text{reg}}(\mathbf{w}, b) = \frac{1}{2n}\lVert\mathbf{X}\mathbf{w} + b\mathbf{1} - \mathbf{y}\rVert^2 + \frac{\lambda}{2n}\lVert\mathbf{w}\rVert^2,$$

with the intercept $b$ **not** penalised. The features are already on comparable scales.

In [ ]:
# 6.1 (5 points): Write ridge_obj(theta, lam) and ridge_grad(theta, lam) for theta = (b, w_1, ..., w_40) on the TRAINING rows, check
# the gradient with num_grad at a random point for lam = 5 (store the max abs difference in gerr6), and fit, for each lam in lams6,
# the exact minimiser (solve the 41 x 41 linear system) and also gradient descent with momentum (eta = 1 / lambda_max of the
# Hessian, beta = 0.9, 3000 steps) for lam = lams6[3]; store the max abs difference between the two in gd_vs_exact6.

rng6 = np.random.default_rng(6)
n_tr6, n_va6, n_te6, p6 = 60, 60, 1000, 40
Xall6 = rng6.normal(size=(n_tr6 + n_va6 + n_te6, p6))
w_true6 = np.zeros(p6); w_true6[:5] = [2.0, -1.5, 1.0, 0.8, -0.5]          # only five features matter
yall6 = 3.0 + Xall6 @ w_true6 + rng6.normal(0, 1.5, len(Xall6))
Xtr6, ytr6 = Xall6[:n_tr6], yall6[:n_tr6]
Xva6, yva6 = Xall6[n_tr6:n_tr6 + n_va6], yall6[n_tr6:n_tr6 + n_va6]
Xte6, yte6 = Xall6[n_tr6 + n_va6:], yall6[n_tr6 + n_va6:]
Dtr6 = np.column_stack([np.ones(n_tr6), Xtr6])
lams6 = [0.0, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0]

# YOUR CODE HERE
gerr6 = gd_vs_exact6 = None
fits6 = None                      # dict lam -> theta (exact)
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.1
assert gerr6 is not None and gd_vs_exact6 is not None and fits6 is not None, "Set gerr6, gd_vs_exact6 and fits6."
assert gerr6 < 1e-6, "ridge_grad disagrees with the numerical gradient."
assert gd_vs_exact6 < 1e-6, "Gradient descent with momentum should reach the exact ridge solution."
assert set(fits6) == set(lams6), "One exact fit per lambda."
print("Self-check passed.")

In [ ]:
# 6.2 (3 points): Compute training and validation MSE for every lambda, choose lam_best6 on VALIDATION, and only then compute the test
# MSE of the chosen model (test6) and of lambda = 0 (test0_6). Plot training and validation MSE against lambda (log x-axis; use
# lambda + 0.1 to show lambda = 0).

# YOUR CODE HERE
lam_best6 = test6 = test0_6 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 6.2
assert lam_best6 is not None and test6 is not None and test0_6 is not None, "Set lam_best6, test6 and test0_6."
assert lam_best6 > 0 and test6 < test0_6, "With 60 trials and 40 features, some ridge penalty should help on held-out data."
print("Self-check passed.")

**6.3 (2 points, written response).** The penalty that gives the smallest Hessian condition number is not the one chosen. Why is that not a contradiction? Why must the test set not be used to choose $\lambda$?

**Your answer:**

*(replace this line with your answer)*

---
## Exercise 7 — Real Data: Fitting a Speed-Tuning Curve by Gradient Descent (15 points)

Lessons 8 and 11 fitted the saturating speed-tuning model $r(s) = r_0 + A\,(1 - e^{-s/s_0})$ to motor-cortex neuron 120 ([Stevenson et al., 2011](https://doi.org/10.1152/jn.00626.2010); [DREAM on CRCNS](https://crcns.org/data-sets/movements/dream); packaged by [Benjamin et al., 2018](https://github.com/KordingLab/spykesML)): $r_0$ and $A$ in spikes/s, $s_0$ in cm/s (valid range $s_0 > 0$). Here you fit it **yourself** by gradient descent, on the objective

$$L(r_0, A, s_0) = \frac{1}{2n}\sum_t\big(r(s_t) - y_t\big)^2\qquad\text{((spikes/s)}^2\text{)},$$

using the first 40% of the session for training and the next 10% for validation. The cell below prepares the data. Do not modify it.

In [ ]:
# PROVIDED -- do not modify.
import os, urllib.request
from scipy.io import loadmat
from scipy.optimize import curve_fit

M1_FILE = 'm1_stevenson_2011.mat'
if not os.path.exists(M1_FILE):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/KordingLab/spykesML/ff2b1830ae07909e5f1a0811445ca3fd5732dc6f/data/' + M1_FILE, M1_FILE)
m1 = loadmat(M1_FILE)
rates7 = m1['spikes'].T.astype(float) / 0.05                     # (15536, 171) spikes/s
vel7 = m1['handVel'][:2].T * 100                                 # (15536, 2) cm/s
speed7 = np.linalg.norm(vel7, axis=1)[2:]                        # (15534,) hand speed 100 ms after each rate bin, cm/s
y7 = rates7[:-2, 120]                                            # (15534,) neuron 120, spikes/s
n7 = len(y7); idx7 = np.arange(n7)
tr7 = idx7 < int(0.4 * n7); va7 = (idx7 >= int(0.4 * n7)) & (idx7 < n7 // 2)
s_tr7, y_tr7, s_va7, y_va7 = speed7[tr7], y7[tr7], speed7[va7], y7[va7]
print("training bins:", tr7.sum(), " validation bins:", va7.sum())

In [ ]:
# 7.1 (5 points): Write loss7(theta, s=s_tr7, y=y_tr7) and grad7(theta) (analytic, training data) for theta = (r0, A, s0). Check
# grad7 against num_grad at theta = (40, 50, 12) (store the max relative error in gerr7). Obtain an independent reference fit with
# scipy's curve_fit (p0 = [45, 60, 10]) and store it in theta_ref7, and the Hessian at theta_ref7 (central differences of grad7,
# step 1e-4 per parameter) in H7 with its eigenvalues in ev7.

# YOUR CODE HERE
gerr7 = theta_ref7 = H7 = ev7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.1
assert gerr7 is not None and theta_ref7 is not None and H7 is not None and ev7 is not None, "Set gerr7, theta_ref7, H7 and ev7."
assert gerr7 < 1e-5, "grad7 disagrees with the numerical gradient."
assert np.linalg.norm(grad7(theta_ref7)) < 1e-3 and np.all(ev7 > 0), "The reference should be a minimum (zero gradient, positive curvature)."
print("Self-check passed.")

In [ ]:
# 7.2 (4 points): Run plain gradient descent (gd_run, eta = 1 / ev7.max(), max_steps 20000, grad_tol 1e-3) from the three starts in
# starts7. For each, store in runs7[i] a dict with the stop reason, iterations, final theta, final training loss, and whether the
# end point is VALID (finite and s0 > 0) and close to the reference (|theta - theta_ref7| < 1 in every component).

starts7 = [np.array([30.0, 40.0, 5.0]), np.array([60.0, 100.0, 25.0]), np.array([40.0, 80.0, 15.0])]

# YOUR CODE HERE
runs7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.2
assert runs7 is not None and len(runs7) == 3, "Set runs7 with three runs."
assert sum(r['close'] for r in runs7) >= 1, "At least one start should reach the reference."
assert any(not r['valid'] or not r['close'] for r in runs7), "At least one start should end somewhere misleading."
print("Self-check passed.")

In [ ]:
# 7.3 (3 points): Diagonal (Jacobi) scaling: theta = D z with D = diag(1 / sqrt(diag(H7))). Store the condition number of D^T H7 D in
# kappa_z7 (compare with ev7). From the good starts (those that reached the reference in 7.2), run GD in z with eta = 1 / lambda_max(D^T H7 D)
# and grad_tol 1e-3 / max(diag(D)) (an equivalent tolerance), and store the iteration counts (theta-run, z-run) in steps7. Finally,
# from starts7[1], run plain GD with the LOOSE tolerance 1e-1 and store loose7 = (theta_loose, train-loss difference from the reference,
# validation-loss difference from the reference).

# YOUR CODE HERE
kappa_z7 = steps7 = loose7 = None
raise NotImplementedError

In [ ]:
# SELF-CHECK 7.3
assert kappa_z7 is not None and steps7 is not None and loose7 is not None, "Set kappa_z7, steps7 and loose7."
assert kappa_z7 < ev7[-1] / ev7[0], "Jacobi scaling should reduce the condition number here."
assert all(z < t for t, z in steps7), "Scaled runs should need fewer iterations."
assert abs(loose7[1]) < 1.0 and np.max(np.abs(loose7[0] - theta_ref7)) > 1.0, "A loose tolerance: tiny loss difference, visibly different parameters."
print("Self-check passed.")

**7.4 (3 points, written response — interpretation and limitation).** Separate three kinds of conclusion. *Optimisation*: what went wrong from the first start, and how would you detect it without knowing the reference answer? What did scaling change? *Prediction*: does the loosely converged fit predict worse? *Science*: what can and cannot be said about $A$ and $s_0$ for this neuron? Give one limitation.

**Your answer:**

*(replace this line with your answer)*

---
## Bonus Challenge — A Backtracking Line Search (extra credit, not part of the 100 points)

The failure from the first start in Exercise 7 came from a step that was safe near the minimum but not far from it. A **backtracking (Armijo) line search** chooses the step afresh at every iteration: start from $\eta = \eta_0$ and halve it until $L(\boldsymbol\theta - \eta\nabla L) \le L(\boldsymbol\theta) - c\,\eta\lVert\nabla L\rVert^2$ (with, say, $c = 10^{-4}$), then take the step.

1. Implement it and rerun the three starts of Exercise 7 with $\eta_0 = 1$. Do all of them now reach the reference?
2. Compare iteration counts and the number of objective evaluations with the fixed-step runs.

**Your code:**

In [ ]:
# YOUR CODE HERE (bonus, optional)
raise NotImplementedError

**Your written answer (bonus):**

*(replace this line with your answer)*

---
## Final Submission Checklist

- All `# YOUR CODE HERE` cells have working code (no `raise NotImplementedError` left in).
- All **Written response** cells contain your own answer, not the placeholder text.
- All self-check cells run without an `AssertionError`.
- All figures have a title, axis labels with units, a legend where appropriate, and no background grid; parameter-space contour plots use equal aspect.
- Every optimiser run you report states its stop reason; nothing that merely ran out of iterations is called converged.
- You ran the whole notebook top to bottom at least once before submitting (Restart & Run All), and no cell raised an unexpected error.